# Lesson 07 - Planning Design Pattern

This notebook demonstrates the **Planning Design Pattern** for AI agents using the Microsoft Agent Framework.
You will learn how to break complex travel requests into structured subtasks, assign them to specialist agents,
and execute the resulting plan — all using structured output powered by Pydantic models.


## സെറ്റപ്പ്


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os, asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## പ്രവൃത്തി വിഭജനo

പ്രവൃത്തി വിഭജനo പ്ലാനിംഗ് ഡിസൈൻ പാറ്റേണിന്റെ കേന്ദ്രഭാഗമാണ്. ഒരു ഏക ഏജന്റിനോട്
സമഗ്രമായ ഒരു ആവശ്യകത ആവിഷ്‌ക്കരിക്കാൻ ആവശ്യപ്പെടുന്നതിനുപകരം, പ്രശ്നം ചെറുതായി, വ്യക്തമാക്കിയ **സബ്‌ടാസ്കുകളായി** വിഭജിക്കുന്നു.
ഓരോ സബ്‌ടാസ്കും ഒരു വിദഗ്ധ ഏജന്റിനാണ് (ഉദാഹരണത്തിന്, ഫ്ലൈറ്റുകൾ, ഹോട്ടലുകൾ, പ്രവർത്തനങ്ങൾ) ദിയ
മുൻഗണനകളും ആശ്രിത ഓർഡറിങ്ങും ഉൾപ്പെടുത്തി നിയോഗിക്കപ്പെടുന്നത്.

ഈ സമീപനം പല ഗുണഫലങ്ങളും നൽകുന്നു:
- **വ്യക്തത**: ഓരോ സബ്‌ടാസ്കിനും ഒരു ഏകദേശ ഉത്തരവാദിത്വമാണ്.
- **സഹകരണപ്രവർത്തനം**: സ്വതന്ത്ര സബ്‌ടാസ്കുകൾ സമാന്തരമായി പ്രവർത്തിക്കാം.
- **വിശ്വാസ്യത**: പരാജയങ്ങൾ വ്യക്തമായ സബ്‌ടാസ്കുകളിലേക്ക് മാത്രം പരിമിതമാക്കപ്പെടുന്നു.
- **ബഡ്ജറ്റ് ട്രാക്കിങ്**: ഓരോ സബ്‌ടാസ്കിന്റെയും ചെലവുകൾ കണക്കുകൂട്ടി സംയോജിപ്പിക്കും.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## ഘടനാബദ്ധമായ ഔട്ട്പുട്ടുമായി ഒരു പ്ലാനിംഗ് ഏജന്റ് സൃഷ്ടിക്കുന്നു

പ്ലാനിംഗ് ഏജന്റ് ഒരു **ഫ്രണ്ട് ഡെസ്ക് കോഓർഡിനേറ്റർ** ആയി പ്രവർത്തിക്കുന്നു. ഉയർന്ന നിലയിലെ യാത്രാ അഭ്യർത്ഥന ലഭിക്കുമ്പോൾ
അത് ഒരു ഘടനാബദ്ധമായ `TravelPlan` ഒരുക്കുന്നു — അഭ്യർത്ഥനയെ ഉപകാര്യങ്ങളായി വിഭജിച്ച്, പ്രാധാന്യതകൾ നിശ്ചയിച്ച്,
ഒരു കോണ്‍സിയർജ് അല്ലെങ്കിൽ എക്സിക്യൂഷൻ ലെയർ പ്രവർത്തനം നടത്താൻ ലിസ്റ്റ് ചെയ്യേണ്ട ആശ്രിതത്വങ്ങൾ തിരിച്ചറിയുന്നു.


In [ ]:
planning_agent = client.as_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    options={"response_format": TravelPlan}
)
if result:
    plan = result.value
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## വിദഗ്ധ ഉപകരണങ്ങളിലൂടെ ഒരു പദ്ധതിയ്ക്ക് ക്രിയാന്വേഷണം ചെയ്യല്‍

ഫ്‌റൺറ് ഡെസ്ക് ഏജന്റ് ഒരു ഘടിതമായ പദ്ധതി ഒരുക്കിയശേഷം, **കൺസിയർജ് ഏജന്റ്** അത് നടപ്പിലാക്കുന്നു.
ഓരോ വിദഗ്ധ ഉപകരണവും ഒരു ഉപകാര്യ വിഭാഗം കൈകാര്യം ചെയ്യുന്നു (ഫ്ലൈറുകൾ, ഹോട്ടലുകൾ, പ്രവർത്തനങ്ങൾ). കൺസിയർജ്
പദ്ധതിയുടെ ഉപകാര്യങ്ങളെ ആശ്രിതക്രമത്തിൽ പരിഗണിച്ച് ഓരോന്നും അനുയോജ്യമായ ഉപകരണത്തിലേക്ക് അയക്കുന്നു.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = client.as_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Summary

In this lesson you learned the **Planning Design Pattern** for AI agents:

1. **Task Decomposition** — A front desk planning agent breaks a complex travel request into
   structured subtasks using Pydantic models, assigning each to a specialist agent with priorities
   and dependencies.
2. **Structured Output** — By passing a `response_format` the agent returns a validated
   `TravelPlan` object instead of free-form text, making downstream processing reliable.
3. **Plan Execution** — A concierge agent iterates through the subtasks using specialist tools
   (`book_flight`, `reserve_hotel`, `book_activity`) to carry out the plan and report results.

This pattern separates *what to do* (planning) from *how to do it* (execution), making agents
more modular, testable, and easier to extend.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
